# Task 2: Paper Figures and Tables

This notebook generates publication-ready figures and tables for the VLM trajectory selection paper.

**Figures:**
1. Line plot: ADE vs K for different candidate generation methods
2. Bar chart: ADE comparison across candidate sets (VLM + baselines)
3. Bar chart: Ablation studies
4. Table (CSV): Model comparison

In [ ]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Publication-quality settings (with sensible font fallbacks)
plt.rcParams.update(
    {
        "font.size": 12,
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "DejaVu Sans", "Liberation Sans"],
        "axes.labelsize": 14,
        "axes.titlesize": 14,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
        "legend.fontsize": 10,
        "figure.figsize": (8, 5),
        "figure.dpi": 150,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "axes.grid": True,
        "grid.alpha": 0.25,
    }
)

# Seaborn theme for publication-ready plots
sns.set_theme(
    context="poster",
    style="whitegrid",
    palette="colorblind",
    rc={
        "axes.spines.right": False,
        "axes.spines.top": False,
        "grid.alpha": 0.25,
        "grid.linestyle": "-",
    },
)

sns.set_context(
    "poster",
    font_scale=1,
)

# Output directory for figures
FIG_OUT_DIR = Path("../latex/figures")
FIG_OUT_DIR.mkdir(parents=True, exist_ok=True)

# CSV output for tables
TABLE_OUT_DIR = Path("../latex/source_data")
TABLE_OUT_DIR.mkdir(parents=True, exist_ok=True)

## Configuration Knobs

In [ ]:
# =============================================================================
# CONFIGURATION KNOBS
# =============================================================================

# --- Seaborn Paper-Ready Style ---
# Use a more professional, clean publication-style preset.
sns.set_theme(
    context="paper",
    style="whitegrid",  # Clean but distinguishes grid with light background
    palette="colorblind",  # Universal accessibility
    font="sans-serif",  # Strong default for papers
    rc={
        "axes.spines.right": False,
        "axes.spines.top": False,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "grid.linestyle": "-",
        "axes.labelweight": "medium",
        "axes.titleweight": "semibold",
        "figure.autolayout": True,
        "legend.frameon": False,
        "legend.loc": "best",
        "lines.linewidth": 2,
        "axes.edgecolor": "#333333",
    },
)

# Primary metric for ADE plots
ADE_METRIC = (
    "ade_model_avg"  # Options: 'ade_model_0_5s_avg', 'ade_model_1_0s_avg', 'ade_model_2_0s_avg'
)

# Figure format
FIG_FORMAT = "pdf"  # Options: 'pdf', 'png', 'svg'

# Color scheme (Seaborn palette; consistent across all figures)
# NOTE: Seaborn is imported in the first cell.
_PALETTE = sns.color_palette("colorblind", 12).as_hex()
COLORS = {
    "topk": _PALETTE[2],
    "nms": _PALETTE[4],
    "kcenter": _PALETTE[3],
    "planner": _PALETTE[1],
    "static": _PALETTE[7],
    "baseline": _PALETTE[5],
    # "planner_v2": _PALETTE[],
    "random": _PALETTE[9],
    "oracle": _PALETTE[0],
}
NEUTRAL_LINE = "0.25"  # dark gray

# Rounded-rectangle bars (applied to all bar charts below)
# Use a radius in *points* (visual units), converted to x/y data-units per-axes.
# This produces true rounded-rectangles (straight sides) even with chunked y-axes.
ROUNDED_BAR_RADIUS_PT = 6


def apply_rounded_bars(ax, bars, *, radius_pt: float = ROUNDED_BAR_RADIUS_PT):
    """Replace rectangular bars with rounded-rectangle PathPatches."""
    from matplotlib.patches import PathPatch
    from matplotlib.path import Path

    fig = ax.figure
    dpi = fig.dpi
    trans = ax.transData

    # Bezier approximation constant for a quarter circle.
    k = 0.5522847498307936

    def rounded_rect_path(x: float, y: float, w: float, h: float, rx: float, ry: float) -> Path:
        rx = max(0.0, min(rx, w * 0.5))
        ry = max(0.0, min(ry, h * 0.5))
        if rx == 0.0 or ry == 0.0:
            verts = [(x, y), (x + w, y), (x + w, y + h), (x, y + h), (x, y)]
            codes = [Path.MOVETO, Path.LINETO, Path.LINETO, Path.LINETO, Path.CLOSEPOLY]
            return Path(verts, codes)

        verts = [
            (x + rx, y),
            (x + w - rx, y),
            (x + w - rx + k * rx, y),
            (x + w, y + ry - k * ry),
            (x + w, y + ry),
            (x + w, y + h - ry),
            (x + w, y + h - ry + k * ry),
            (x + w - rx + k * rx, y + h),
            (x + w - rx, y + h),
            (x + rx, y + h),
            (x + rx - k * rx, y + h),
            (x, y + h - ry + k * ry),
            (x, y + h - ry),
            (x, y + ry),
            (x, y + ry - k * ry),
            (x + rx - k * rx, y),
            (x + rx, y),
        ]
        codes = [
            Path.MOVETO,
            Path.LINETO,
            Path.CURVE4,
            Path.CURVE4,
            Path.CURVE4,
            Path.LINETO,
            Path.CURVE4,
            Path.CURVE4,
            Path.CURVE4,
            Path.LINETO,
            Path.CURVE4,
            Path.CURVE4,
            Path.CURVE4,
            Path.LINETO,
            Path.CURVE4,
            Path.CURVE4,
            Path.CURVE4,
        ]
        # Close the path explicitly for consistent fills.
        verts.append((x + rx, y))
        codes.append(Path.CLOSEPOLY)
        return Path(verts, codes)

    xlim = ax.get_xlim()
    ylim = ax.get_ylim()

    for bar in list(bars):
        x = bar.get_x()
        y = bar.get_y()
        w = bar.get_width()
        h = bar.get_height()
        if w == 0 or h == 0:
            continue

        if h < 0:
            y = y + h
            h = -h

        # Desired radius in pixels.
        r_px = float(radius_pt) * dpi / 72.0

        # Convert to data-units in x/y using local axis scales.
        x0_px, y0_px = trans.transform((x, y))
        x1_px, _ = trans.transform((x + 1.0, y))
        _, y1_px = trans.transform((x, y + 1.0))
        sx = abs(x1_px - x0_px)
        sy = abs(y1_px - y0_px)
        rx = (r_px / sx) if sx else 0.0
        ry = (r_px / sy) if sy else 0.0

        # Clamp radii to the bar size.
        eps = 1e-12
        rx = max(0.0, min(rx, (w * 0.5) - eps))
        ry = max(0.0, min(ry, (h * 0.5) - eps))

        fc = bar.get_facecolor()
        ec = bar.get_edgecolor()
        lw = bar.get_linewidth()
        alpha = bar.get_alpha()
        z = bar.get_zorder()

        bar.remove()

        path = rounded_rect_path(x, y, w, h, rx, ry)
        patch = PathPatch(
            path,
            facecolor=fc,
            edgecolor=ec,
            linewidth=lw,
            alpha=alpha,
            transform=trans,
            zorder=z,
        )
        patch.set_clip_path(ax.patch)
        ax.add_patch(patch)

    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

## Load and Parse Data

In [ ]:
# Load the CSV data
csv_path = Path("../latex/source_data/task2_data.csv")
df_raw = pd.read_csv(csv_path)

# Remove empty rows and comment rows
df = df_raw.dropna(subset=["experiment_name"])
df = df[~df["experiment_name"].str.startswith("--", na=False)]
df = df[~df["experiment_name"].str.startswith("-", na=False)]
df = df.reset_index(drop=True)


# =============================================================================
# Parse experiment names
# =============================================================================
def parse_experiment_name(name: str) -> dict:
    info = {
        "candidate_type": None,
        "K": None,
        "model": None,
        "hist_frames": 0,
        "no_goal": False,
        "show_scores": False,
    }
    name = name.strip()

    # Model
    match = re.search(r"policy=([\w.-]+)", name)
    if match:
        info["model"] = match.group(1)

    # Candidate types
    if m := re.search(r"cand=topk(\d+)", name):
        info["candidate_type"], info["K"] = "topk", int(m.group(1))
    elif m := re.search(r"cand=kcenter(\d+)", name):
        info["candidate_type"], info["K"] = "kcenter", int(m.group(1))
    elif m := re.search(r"cand=nms(\d+)", name):
        info["candidate_type"], info["K"] = "nms", int(m.group(1))
    elif "cand=planner_v2" in name:
        info["candidate_type"], info["K"] = "planner_v2", 12
    elif m := re.search(r"cand=static_(\w+?)(\d+)", name):
        info["candidate_type"] = f"static_{m.group(1)}"
        info["K"] = int(m.group(2))

    # Baselines
    if "policy=dummy_argmax" in name and "topk" not in name:
        info["model"], info["candidate_type"] = "dummy_argmax", "baseline"
    if "policy=dummy_random" in name and "topk" not in name:
        info["model"], info["candidate_type"] = "dummy_random", "baseline"
    if "policy=oracle_min_ade_all" in name:
        info["model"], info["candidate_type"] = "oracle_min_ade_all", "baseline"
    elif "policy=oracle_min_ade" in name:
        info["model"], info["candidate_type"] = "oracle_min_ade", "baseline"
    if "policy=sample_nms_prob" in name:
        info["model"], info["candidate_type"] = "sample_nms_prob", "baseline"

    # Random within topk
    if m := re.search(r"policy=dummy_random_topk(\d+)", name):
        info["model"], info["candidate_type"] = "dummy_random_topk", "random_topk"
        info["K"] = int(m.group(1))

    # Flags
    if m := re.search(r"hist(?:frames)?=(\d+)", name):
        info["hist_frames"] = int(m.group(1))
    info["no_goal"] = "no-goal" in name or "no_goal" in name
    info["show_scores"] = "show-scores" in name or "show_scores" in name

    return info


parsed = df["experiment_name"].apply(parse_experiment_name).apply(pd.Series)
df = pd.concat([df, parsed], axis=1)

print(f"Loaded {len(df)} experiments")
print(f"Unique candidate_type: {df['candidate_type'].dropna().unique().tolist()}")
print(f"Unique model: {df['model'].dropna().unique().tolist()}")

## Experiment Browser

Use this to explore and find exact experiment names for plotting.

In [ ]:
# Show all experiments with key columns
cols = ["experiment_name", "selector", "candidate_type", "K", "model", "no_goal", "ade_model_avg"]
cols = [c for c in cols if c in df.columns]
display(df[cols].sort_values("ade_model_avg"))

In [ ]:
# Print all experiment names, fully and without truncation
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    print(df["experiment_name"])

## Helper Functions

In [ ]:
def find_exp(df: pd.DataFrame, substring: str) -> pd.Series:
    """Find ONE experiment matching substring.

    If multiple matches exist (duplicates or substring collisions), pick a deterministic
    best match:
      1) exact `experiment_name == substring`
      2) exact `experiment_name == 'ANCHOR_' + substring`
      3) latest `created_at_utc` (if available)
      4) first match (fallback)
    """
    matches = df[df["experiment_name"].str.contains(substring, na=False, regex=False)]
    if len(matches) == 0:
        raise ValueError(f"No match: '{substring}'")
    if len(matches) == 1:
        return matches.iloc[0]

    exact = matches[matches["experiment_name"] == substring]
    if len(exact) == 1:
        return exact.iloc[0]

    anchor_name = substring if substring.startswith("ANCHOR_") else f"ANCHOR_{substring}"
    anchor = matches[matches["experiment_name"] == anchor_name]
    if len(anchor) == 1:
        return anchor.iloc[0]

    chosen = None
    chosen_reason = None

    if "created_at_utc" in matches.columns:
        ts = pd.to_datetime(matches["created_at_utc"], errors="coerce")
        if ts.notna().any():
            chosen_idx = ts.idxmax()
            chosen = df.loc[chosen_idx]
            chosen_reason = f"latest created_at_utc={chosen.get('created_at_utc')}"

    if chosen is None:
        chosen = matches.iloc[0]
        chosen_reason = "first match (fallback)"

    print(
        f"WARNING: Multiple matches ({len(matches)}) for '{substring}'. "
        f"Using {chosen_reason}:\n"
        + "\n".join(f"  - {x}" for x in matches["experiment_name"].tolist())
    )
    return chosen


def get_metric(df, exp_name, metric="ade_model_avg"):
    """Get metric value for an experiment by name substring."""
    return find_exp(df, exp_name)[metric]

## Figure 1: ADE vs K (Line Plot)

Configure exactly which experiments to plot for each line.

In [ ]:
_PALETTE

In [ ]:
# =============================================================================
# FIGURE 1 CONFIGURATION: Define which experiments to plot
# Format: { "line_label": [(K, "experiment_name_substring"), ...] }
# =============================================================================

FIG1_LINES = {
    "VLM TopK (w/o goal)": [
        (12, "0122_T2_real_policy=gemini-3-flash_cand=topk12_no-goal"),
        (18, "ANCHOR_0120_T2_real_policy=gemini-3-flash_cand=topk18_no-goal"),
        (24, "0120_T2_real_policy=gemini-3-flash_cand=topk24_no-goal-in-image"),
    ],
    "VLM TopK (w/ goal)": [
        (6, "0118_T2_real_policy=gemini-3-flash_cand=topk6_night-visual-prompting"),
        (12, "0119_T2_real_policy=gemini-3-flash_cand=topk12_night-visual-prompting"),
        (18, "0119_T2_real_policy=gemini-3-flash_cand=topk18_night-visual-prompting"),
        (24, "0119_T2_real_policy=gemini-3-flash_cand=topk24_night-visual-prompting"),
        (30, "0119_T2_real_policy=gemini-3-flash_cand=topk30_night-visual-prompting"),
    ],
    "VLM K-Center (w/ goal)": [
        (6, "0118_T2_real_policy=gemini-3-flash_cand=kcenter6_night-visual-prompting"),
        (12, "0118_T2_real_policy=gemini-3-flash_cand=kcenter12_night-visual-prompting"),
        (18, "0118_T2_real_policy=gemini-3-flash_cand=kcenter18_night-visual-prompting"),
        (24, "0118_T2_real_policy=gemini-3-flash_cand=kcenter24_night-visual-prompting"),
    ],
    # "VLM Static KMeans (w/ goal)": [
    #     (6,  "0119_T2_real_policy=gemini-3-flash_cand=static_kmeans6_takeover"),
    #     (12, "0119_T2_real_policy=gemini-3-flash_cand=static_kmeans12_takeover"),
    #     (18, "0119_T2_real_policy=gemini-3-flash_cand=static_kmeans18_takeover"),
    # ],
    "Random (TopK)": [
        (6, "0119_T2_real_policy=dummy_random_topk6"),
        (12, "0119_T2_real_policy=dummy_random_topk12"),
        (18, "0119_T2_real_policy=dummy_random_topk18"),
        (24, "0119_T2_real_policy=dummy_random_topk24"),
        (30, "0119_T2_real_policy=dummy_random_topk30"),
    ],
    # "Oracle (Argmin ADE)": [
    #     (6, "0115_T2_real_policy=oracle_min_ade_pool=raw_topk_k=6"),
    #     (12, "0115_T2_real_policy=oracle_min_ade_pool=raw_topk_k=12"),
    #     (18, "0115_T2_real_policy=oracle_min_ade_pool=raw_topk_k=18"),
    #     (24, "0115_T2_real_policy=oracle_min_ade_pool=raw_topk_k=24"),
    #     (30, "0115_T2_real_policy=oracle_min_ade_pool=raw_topk_k=30"),
    # ]
}

FIG1_BASELINES = {
    "Planner": "0118_T2_real_policy=dummy_argmax",
    # "Planner (Argmin ADE)": "0115_T2_real_policy=oracle_min_ade_pool=planner_v2_k=6",
}

FIG1_STYLES = {
    "VLM TopK (w/ goal)": {"color": _PALETTE[6], "marker": "o"},
    "VLM TopK (w/o goal)": {"color": COLORS["topk"], "marker": "o"},
    "VLM K-Center (w/ goal)": {"color": COLORS["kcenter"], "marker": "s"},
    "VLM Static KMeans (w/ goal)": {"color": COLORS["static"], "marker": "D"},
    "Random (TopK)": {"color": COLORS["random"], "marker": "x", "linestyle": "--"},
}

In [ ]:
# Generate Figure 1
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(5, 4))

print("FIGURE 1: Experiments used")
print("=" * 80)

color_keys = []
colors = []

for label, experiments in FIG1_LINES.items():
    ks, ades = [], []
    print(f"\n[{label}]")
    for k, exp_name in experiments:
        try:
            row = find_exp(df, exp_name)
            ks.append(k)
            ades.append(row[ADE_METRIC])
            print(f"  K={k:2d}: {ADE_METRIC}={row[ADE_METRIC]:.4f}  <- {row['experiment_name']}")
        except ValueError as e:
            print(f"  WARNING: {e}")

    if ks:
        style = FIG1_STYLES.get(label, {})
        ax.plot(
            ks,
            ades,
            marker=style.get("marker", "o"),
            markersize=8,
            linewidth=2,
            color=style.get("color", NEUTRAL_LINE),
            linestyle=style.get("linestyle", "-"),
            label=label,
        )
        # Track color for legend
        color_key = None
        # Using the color key as appears in FIG1_STYLES if possible, else label itself
        for k, _v in FIG1_STYLES.items():
            if label == k:
                color_key = k
                break
        if color_key is None:
            color_key = label
        color_keys.append(color_key)
        colors.append(style.get("color", NEUTRAL_LINE))

print("\n[BASELINES]")
for label, exp_name in FIG1_BASELINES.items():
    try:
        row = find_exp(df, exp_name)
        baseline_color = COLORS["planner"] if "Planner" in label else COLORS["oracle"]
        ax.axhline(
            y=row[ADE_METRIC],
            linestyle=":",
            linewidth=2,
            color=baseline_color,
            label=label,
        )
        print(f"  {label}: {row[ADE_METRIC]:.4f}  <- {row['experiment_name']}")
        # Add baseline to color legend if desired
        color_keys.append(label)
        colors.append(baseline_color)
    except ValueError as e:
        print(f"  WARNING: {e}")

ax.set_xlabel("Number of Candidates (K)")
ax.set_ylabel("ADE (m)")
# ax.set_title("VLM Trajectory Selection: ADE vs Candidate Set Size")

# ---- Beautiful Legend Section (copied and lightly adapted from persistent legend code) ----
# Collect unique color_key, color pairs only for those keys/colors actually used
used_legend_items = []
seen_keys = set()
for k, c in zip(color_keys, colors, strict=False):
    if k not in seen_keys:
        used_legend_items.append((k, c))
        seen_keys.add(k)

# Pretty label mapping for legend (customize as necessary)
PRETTY_LABELS = {
    "static": "Static Candidates",
    "VLM TopK (w/ goal)": "TopK (VLM, w/ goal)",
    "VLM TopK (w/o goal)": "TopK (VLM, no goal)",
    "VLM K-Center (w/ goal)": "K-Center (VLM, w/ goal)",
    "VLM Static KMeans (w/ goal)": "KMeans (VLM, w/ goal)",
    "Random (TopK)": "Random TopK",
    "Planner": "Planner Argmax",
    # Add as appropriate for your keys
    "oracle": "Oracle",
    "nms": "NMS",
    "planner": "Planner Argmax",
    "random": "Random",
    # You can add more labels if needed
}
legend_handles = [
    mpatches.Patch(color=color, label=PRETTY_LABELS.get(key, key))
    for key, color in used_legend_items
    if key in PRETTY_LABELS or "Planner" in key or "Oracle" in key
]

ax.legend(handles=legend_handles, loc="center right", title=None, fontsize=9, title_fontsize=10)
# ---------------------------------------------

sns.despine(ax=ax)
plt.tight_layout()
out_path = FIG_OUT_DIR / f"fig1_ade_vs_k.{FIG_FORMAT}"
fig.savefig(out_path)
print(f"\nSaved: {out_path}")
plt.show()

## Figure 2: Bar Chart - Candidate Comparison

Includes VLM results AND baselines (Oracle, Planner, NMS-sampled, etc.)

In [ ]:
# =============================================================================
# FIGURE 2 CONFIGURATION
# Format: [(display_label, experiment_substring, color_key), ...]
# =============================================================================

FIG2_BARS = [
    # Baselines first
    # ("Oracle\n(Min ADE)", "0118_T2_real_policy=oracle_min_ade", "oracle"),
    # ("Oracle\n(Min ADE All)", "0118_T2_real_policy=oracle_min_ade_all", "oracle"),
    # ("Planner\nArgmax", "0118_T2_real_policy=dummy_argmax", "planner"),
    # ("Sample\nNMS-Prob", "0118_T2_real_policy=sample_nms_prob", "baseline"),
    # ("Random", "0118_T2_real_policy=dummy_random", "random"),
    # VLM results
    # ("VLM TopK-12\n(no-goal)", "0122_T2_real_policy=gemini-3-flash_cand=topk12_no-goal", "topk"),
    # ("VLM TopK-18\n(w/o goal)", "0120_T2_real_policy=gemini-3-flash_cand=topk18_no-goal", "topk"),
    ("Top-K", "0119_T2_real_policy=gemini-3-flash_cand=topk24_night-visual-prompting", "topk"),
    (
        "K-center",
        "0118_T2_real_policy=gemini-3-flash_cand=kcenter18_night-visual-prompting",
        "kcenter",
    ),
    ("post-NMS", "0118_T2_real_policy=gemini-3-flash_cand=nms6_night-visual-prompting", "nms"),
    (
        "post-NMS\n(+prob. filter)",
        "0118_T2_real_policy=gemini-3-flash_cand=planner_v2_night-visual-prompting",
        "planner",
    ),
    (
        "Chain-of-\nPlanning",
        "0121_T2_real_policy=gemini-3-flash_cand=chain-of-planning64_b4_l4_leaf8_no-chat-history_with-goal",
        "XXXX",
    ),
    (
        "Grid\n(K=12)",
        "0119_T2_real_policy=gemini-3-flash_cand=static_grid12_nearest_real_takeover",
        "static",
    ),
    (
        "K-disk\n(K=12)",
        "0119_T2_real_policy=gemini-3-flash_cand=static_kdisk_endpoints12_takeover",
        "static",
    ),
    (
        "K-means\n(K=12)",
        "0119_T2_real_policy=gemini-3-flash_cand=static_kmeans12_takeover",
        "static",
    ),
]

In [ ]:
# Generate Figure 2
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(8, 4))

print("FIGURE 2: Experiments used")
print("=" * 80)

names, values, colors = [], [], []
# Also store the color_key for legend
color_keys = []
for label, exp_name, color_key in FIG2_BARS:
    # try:
    row = find_exp(df, exp_name)
    names.append(label)
    values.append(row[ADE_METRIC])
    color = COLORS.get(color_key, NEUTRAL_LINE)
    colors.append(color)
    color_keys.append(color_key)
    print(f"  {label.replace(chr(10), ' ')}: {row[ADE_METRIC]:.4f}  <- {row['experiment_name']}")
    # except ValueError as e:
    #     print(f"  WARNING: {e}")

x = np.arange(len(names))
bars = ax.bar(x, values, color=colors)

# ----------------------
# Create legend by color
# ----------------------
# Collect unique color_key, color pairs only for those keys/colors actually used
used_legend_items = []
seen_keys = set()
for k, c in zip(color_keys, colors, strict=False):
    if k not in seen_keys:
        used_legend_items.append((k, c))
        seen_keys.add(k)

# Pretty label mapping for legend (customize as necessary)
# PRETTY_LABELS = {
#     # "oracle": "Oracle",
#     # "planner": "Planner Argmax",
#     # "planner_v2": "Prob. Filter",
#     # "nms": "NMS",
#     # "topk": "TopK",
#     # "kcenter": "K-center",
#     "static": "Static Candidates",
#     # "random": "Random",
#     # "baseline": "Baseline",
# }
# legend_handles = [
#     mpatches.Patch(color=color, label=PRETTY_LABELS.get(key, key))
#     for key, color in used_legend_items if key in PRETTY_LABELS
# ]
# ax.legend(handles=legend_handles, loc='upper right', title=None, fontsize=9, title_fontsize=10)

# Example: show all legend keys/labels, rotating legend text by 30 degrees
# legend_handles = [
#     mpatches.Patch(color=color, label=key)
#     for key, color in used_legend_items
# ]
# leg = ax.legend(handles=legend_handles, loc='upper right', title=None,
#                 fontsize=9, title_fontsize=10)
# for text in leg.get_texts():
#     text.set_rotation(30)

# Custom y-limit
ylim_low = 1.0
ylim_high = 1.65
ax.set_ylim(bottom=ylim_low, top=ylim_high)
overflow_offset = 0.1  # vertical offset above y-limit for out-of-range labels

for bar, val in zip(bars, values, strict=False):
    text_y = bar.get_height() + 0.02
    if val > ylim_high:
        text_y = ylim_high - 0.02 - overflow_offset
        ax.annotate(
            f"{val:.3f}",
            xy=(bar.get_x() + bar.get_width() / 2, ylim_high - 0.04),
            xytext=(bar.get_x() + bar.get_width() / 2, text_y),
            ha="center",
            va="bottom",
            fontsize=9,
            arrowprops=dict(arrowstyle="->", color="gray", lw=1),
            bbox=dict(boxstyle="round,pad=0.2", fc="w", ec="gray", alpha=0.8),
        )
    else:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            text_y,
            f"{val:.3f}",
            ha="center",
            va="bottom",
            fontsize=9,
        )

ax.set_xticks(x)
ax.set_xticklabels(names, fontsize=9)
ax.set_ylabel("ADE (m)")
# ax.set_title("Trajectory Selection: Comparison of Candidate Sets")
apply_rounded_bars(ax, bars)
sns.despine(ax=ax)
plt.tight_layout()
out_path = FIG_OUT_DIR / f"fig2_comparison.{FIG_FORMAT}"
fig.savefig(out_path)
print(f"\nSaved: {out_path}")
plt.show()

In [ ]:
# =============================================================================
# FIGURE 2 B  CONFIGURATION
# Format: [(display_label, experiment_substring, color_key), ...]
# =============================================================================

FIG2_BARS = [
    # Baselines first
    ("Oracle\n(All Candidates)", "0118_T2_real_policy=oracle_min_ade_all", "oracle"),
    # ("Oracle\n(Min ADE)", "0118_T2_real_policy=oracle_min_ade", "oracle"),
    (
        "Oracle\n(Planner's Candidates)",
        "0115_T2_real_policy=oracle_min_ade_pool=planner_v2_k=6",
        "oracle",
    ),
    ("Planner\n(Argmax Score)", "0118_T2_real_policy=dummy_argmax", "planner"),
    # ("Sample\nNMS-Prob", "0118_T2_real_policy=sample_nms_prob", "baseline"),
    # ("Random", "0118_T2_real_policy=dummy_random", "random"),
    # VLM results
    # ("VLM TopK-12\n(no-goal)", "0122_T2_real_policy=gemini-3-flash_cand=topk12_no-goal", "topk"),
    (
        "VLM Top-K\n(Ours, w/o goal)",
        "0120_T2_real_policy=gemini-3-flash_cand=topk18_no-goal",
        "topk",
    ),
    # (
    #     "VLM KCenter-18",
    #     "0118_T2_real_policy=gemini-3-flash_cand=kcenter18_night-visual-prompting",
    #     "kcenter",
    # ),
    # ("VLM post-NMS",
    #  "0118_T2_real_policy=gemini-3-flash_cand=nms6_night-visual-prompting", "nms"),
    # ("VLM post-NMS (prob. filter)",
    #  "0118_T2_real_policy=gemini-3-flash_cand=planner_v2_night-visual-prompting",
    #  "planner_v2"),
]

# Generate Figure 2
fig, ax = plt.subplots(figsize=(6, 4))

print("FIGURE 2: Experiments used")
print("=" * 80)

names, values, colors = [], [], []
for label, exp_name, color_key in FIG2_BARS:
    try:
        row = find_exp(df, exp_name)
        names.append(label)
        values.append(row[ADE_METRIC])
        colors.append(COLORS.get(color_key, NEUTRAL_LINE))
        print(
            f"  {label.replace(chr(10), ' ')}: {row[ADE_METRIC]:.4f}  <- {row['experiment_name']}"
        )
    except ValueError as e:
        print(f"  WARNING: {e}")

x = np.arange(len(names))
bars = ax.bar(x, values, color=colors, edgecolor="black", linewidth=0.0)

# Identify the index to bold (for "VLM Top-K\n(Ours, w/o goal)")
bold_label = "VLM Top-K\n(Ours, w/o goal)"
bold_idx = None
for idx, lbl in enumerate(names):
    if lbl == bold_label:
        bold_idx = idx
        break

for _i, (bar, val) in enumerate(zip(bars, values, strict=False)):
    fontweight = "normal"
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.02,
        f"{val:.3f}",
        ha="center",
        va="bottom",
        fontsize=9,
        fontweight=fontweight,
    )

ax.set_xticks(x)

# Now, make the x-axis label bold for "VLM Top-K\n(Ours, w/o goal)"
xtick_fontsizes = [9 for _ in names]
xtick_fontweights = ["bold" if i == bold_idx else "normal" for i in range(len(names))]

# Set the x-tick labels with per-label fontweight
for _tick, label, weight in zip(
    ax.set_xticks(x), ax.set_xticklabels(names, fontsize=9), xtick_fontweights, strict=False
):
    label.set_fontweight(weight)

ax.set_ylabel("ADE (m)")
# ax.set_title("Trajectory Selection: Comparison of Methods")
ax.set_ylim(bottom=0)
ax.set_axisbelow(True)
apply_rounded_bars(ax, bars)
sns.despine(ax=ax)
plt.tight_layout()
out_path = FIG_OUT_DIR / f"fig2b_comparison_V2.{FIG_FORMAT}"
fig.savefig(out_path)
print(f"\nSaved: {out_path}")
plt.show()

## Figure 3: Ablation Study

Compare effects of history frames, goal visibility, score visibility.

In [ ]:
# =============================================================================
# FIGURE 3: ABLATION EXPERIMENTS
# =============================================================================

# History frames ablation (topk18, no-goal)
ABLATION_HISTORY = [
    ("0 frames", "0120_T2_real_policy=gemini-3-flash_cand=topk18_no-goal"),
    ("1 frame", "0121_T2_real_policy=gemini-3-flash_cand=topk18_no-goal_hist=1"),
    ("2 frames", "0121_T2_real_policy=gemini-3-flash_cand=topk18_no-goal_hist=2"),
]

# Goal visibility ablation (kcenter18)
ABLATION_GOAL = [
    ("No Goal\nNo Scores", "0120_T2_real_policy=gemini-3-flash_cand=kcenter18_no-goal"),
    (
        "With Goal\nNo Scores",
        "0118_T2_real_policy=gemini-3-flash_cand=kcenter18_night-visual-prompting",
    ),
    (
        "W/ Goal\nW/ Scores",
        "0119_T2_real_policy=gemini-3-flash_cand=kcenter18_night-visual-prompting_show-scores",
    ),
]

In [ ]:
# Generate Figure 3a: History Frames Ablation (with chunked/broken axis)
fig_hist, ax_hist = plt.subplots(figsize=(3, 2.5))

print("FIGURE 3a: History Frames Ablation")
print("=" * 80)

names, values = [], []
print("\n[History Frames Ablation]")
for label, exp_name in ABLATION_HISTORY:
    try:
        row = find_exp(df, exp_name)
        ade_val = float(row[ADE_METRIC])
        names.append(label)
        values.append(ade_val)
        print(f"  {label}: {ade_val:.4f}  <- {row['experiment_name']}")
    except (ValueError, TypeError) as e:
        print(f"  WARNING: {e}")
y_bottom = 1.0
if names:
    # Find a suitable bottom for the y-axis to "chunk" the bar plot (do not start from zero)
    min_val, max_val = min(values), max(values)
    # Make a little padding around the min/max range
    pad = (max_val - min_val) * 0.25 if max_val > min_val else 0.10

    y_top = max_val + pad

    x = np.arange(len(names))
    heights = np.array(values) - y_bottom
    bars = ax_hist.bar(x, heights, bottom=y_bottom, color=COLORS["topk"], linewidth=0.0)
    for bar, val in zip(bars, values, strict=False):
        ax_hist.text(
            bar.get_x() + bar.get_width() / 2,
            val + 0.01 * (y_top - y_bottom),
            f"{val:.3f}",
            ha="center",
            va="bottom",
            fontsize=10,
        )
    ax_hist.set_xticks(x)
    ax_hist.set_xticklabels(names)
    ax_hist.set_ylabel("ADE (m)")
    # ax_hist.set_title("Effect of History Frames")
    # Don't start from zero: chunk the axis to min_val - pad
    ax_hist.set_ylim(bottom=y_bottom)
    ax_hist.set_axisbelow(True)
    apply_rounded_bars(ax_hist, bars)

    # # Optionally indicate break in y-axis visually (keep it subtle)
    # d = 0.012  # size of diagonal lines in axes coords
    # kwargs = dict(transform=ax_hist.transAxes, color=NEUTRAL_LINE, lw=1.5, clip_on=False)
    # ax_hist.plot((-d, +d), (-d, +d), **kwargs)        # bottom-left diagonal
    # ax_hist.plot((1 - d, 1 + d), (-d, +d), **kwargs)  # bottom-right diagonal

sns.despine(ax=ax_hist)
plt.tight_layout()
out_path_hist = FIG_OUT_DIR / f"fig3a_ablation_history.{FIG_FORMAT}"
fig_hist.savefig(out_path_hist)
print(f"\nSaved: {out_path_hist}")
plt.show()

# Generate Figure 3b: Goal/Score Visibility Ablation (with chunked/broken axis)
fig_goal, ax_goal = plt.subplots(figsize=(3, 2.5))

print("FIGURE 3b: Goal/Score Visibility Ablation")
print("=" * 80)

names, values = [], []
print("\n[Goal/Score Visibility Ablation]")
for label, exp_name in ABLATION_GOAL:
    try:
        row = find_exp(df, exp_name)
        ade_val = float(row[ADE_METRIC])
        names.append(label)
        values.append(ade_val)
        print(f"  {label}: {ade_val:.4f}  <- {row['experiment_name']}")
    except (ValueError, TypeError) as e:
        print(f"  WARNING: {e}")

if names:
    min_val, max_val = min(values), max(values)
    pad = (max_val - min_val) * 0.25 if max_val > min_val else 0.10

    y_top = max_val + pad

    x = np.arange(len(names))
    heights = np.array(values) - y_bottom
    bars = ax_goal.bar(x, heights, bottom=y_bottom, color=_PALETTE[0], linewidth=0.0)
    for bar, val in zip(bars, values, strict=False):
        ax_goal.text(
            bar.get_x() + bar.get_width() / 2,
            val + 0.01 * (y_top - y_bottom),
            f"{val:.3f}",
            ha="center",
            va="bottom",
            fontsize=10,
        )
    ax_goal.set_xticks(x)
    ax_goal.set_xticklabels(names)
    ax_goal.set_ylabel("ADE (m)")
    # ax_goal.set_title("Effect of Goal/Score Visibility")
    ax_goal.set_ylim(bottom=y_bottom)
    ax_goal.set_axisbelow(True)
    apply_rounded_bars(ax_goal, bars)

    # d = 0.012
    # kwargs = dict(transform=ax_goal.transAxes, color=NEUTRAL_LINE, lw=1.5, clip_on=False)
    # ax_goal.plot((-d, +d), (-d, +d), **kwargs)
    # ax_goal.plot((1 - d, 1 + d), (-d, +d), **kwargs)

sns.despine(ax=ax_goal)
plt.tight_layout()
out_path_goal = FIG_OUT_DIR / f"fig3b_ablation_goal.{FIG_FORMAT}"
fig_goal.savefig(out_path_goal)
print(f"\nSaved: {out_path_goal}")
plt.show()

## Table 4: Model Comparison (CSV)

Compare different VLM models with the same settings (topk18, no-goal).

In [ ]:
# =============================================================================
# TABLE 4: MODEL COMPARISON
# =============================================================================

# (display name, experiment_name substring)
MODEL_COMPARISON = [
    ("Gemini 3 Flash", "0120_T2_real_policy=gemini-3-flash_cand=topk18_no-goal"),
    ("Gemini 2.5 Flash", "0122_T2_real_policy=gemini-2.5-flash_cand=topk18_no-goal"),
    ("Gemini 2.5 Flash Lite", "0122_T2_real_policy=gemini-2.5-flash-lite_cand=topk18_no-goal"),
    (
        "Gemini Robotics ER 1.5",
        "0122_T2_real_policy=gemini-robotics-er-1.5-preview_cand=topk18_no-goal",
    ),
    ("GPT-5 Mini", "0122_T2_real_policy=gpt-5-mini_cand=topk18_no-goal"),
    ("GPT-5 Nano", "0122_T2_real_policy=gpt-5-nano_cand=topk18_no-goal"),
    ("Qwen2.5-VL-72B", "0125_T2_real_policy=qwen2.5vl72b_cand=topk18_no-goal"),
    ("Qwen2.5-VL-32B", "0125_T2_real_policy=qwen2.5vl32b_cand=topk18_no-goal"),
    ("Qwen2.5-VL-7B", "0125_T2_real_policy=qwen2.5vl7b_cand=topk18_no-goal"),
]

TABLE4_BASELINES = [
    ("Planner Argmax", "0118_T2_real_policy=dummy_argmax"),
    ("Oracle (Min ADE)", "0118_T2_real_policy=oracle_min_ade"),
]

# Values we want to put into Table 4 (rounded to 3 decimals)
TABLE4_METRICS_3DP = [
    "ade_model_avg",
    "ade_model_0_5s_avg",
    "ade_model_1_0s_avg",
    "ade_model_2_0s_avg",
    "selected_end_dist_to_goal_m_avg",
    "selected_goal_ang_diff_deg_avg",
    "selected_traj_avg_dist_to_goal_m_avg",
    "selected_goal_progress_m_avg",
    "maoe_deg_avg",
    "dcr_avg",
    "tcr_avg",
]

# Raw stats columns (no processing: copied directly from the CSV)
TABLE4_RAW_STATS_COLS = [
    "model_calls",
    "prompt_tokens_total",
    "output_tokens_total",
    "total_tokens_total",
    "total_tokens_avg",
    "latency_ms_p50",
]

In [ ]:
# Generate Table 4 (robust to running this section standalone)
from pathlib import Path

import pandas as pd

try:
    from IPython.display import display  # type: ignore
except Exception:
    # Fallback for non-notebook execution
    def display(x):
        print(x)


def _load_task2_csv(csv_path: Path) -> pd.DataFrame:
    df_raw = pd.read_csv(csv_path)

    # Remove empty rows and comment rows
    df = df_raw.dropna(subset=["experiment_name"])
    df = df[~df["experiment_name"].astype(str).str.startswith("--", na=False)]
    df = df[~df["experiment_name"].astype(str).str.startswith("-", na=False)]
    return df.reset_index(drop=True)


def _ensure_table4_context() -> None:
    global df, TABLE_OUT_DIR

    def _resolve_task2_csv_path() -> Path:
        candidates = [
            Path("../latex/source_data/task2_data.csv"),
            Path("latex/source_data/task2_data.csv"),
        ]
        for p in candidates:
            if p.exists():
                return p
        tried = ", ".join(str(p.resolve()) for p in candidates)
        raise FileNotFoundError(f"Missing CSV. Tried: {tried}")

    csv_path = _resolve_task2_csv_path()

    if "TABLE_OUT_DIR" not in globals():
        TABLE_OUT_DIR = csv_path.parent
        TABLE_OUT_DIR.mkdir(parents=True, exist_ok=True)

    if "df" not in globals():
        df = _load_task2_csv(csv_path)

    if "find_exp" not in globals():

        def find_exp(df: pd.DataFrame, substring: str) -> pd.Series:
            """Find ONE experiment matching substring.

            If multiple matches exist (duplicates or substring collisions), pick a deterministic
            best match:
              1) exact `experiment_name == substring`
              2) exact `experiment_name == 'ANCHOR_' + substring`
              3) latest `created_at_utc` (if available)
              4) first match (fallback)
            """
            matches = df[
                df["experiment_name"].astype(str).str.contains(substring, na=False, regex=False)
            ]
            if len(matches) == 0:
                raise ValueError(f"No match: '{substring}'")
            if len(matches) == 1:
                return matches.iloc[0]

            exact = matches[matches["experiment_name"] == substring]
            if len(exact) == 1:
                return exact.iloc[0]

            anchor_name = substring if substring.startswith("ANCHOR_") else f"ANCHOR_{substring}"
            anchor = matches[matches["experiment_name"] == anchor_name]
            if len(anchor) == 1:
                return anchor.iloc[0]

            chosen = None
            chosen_reason = None

            if "created_at_utc" in matches.columns:
                ts = pd.to_datetime(matches["created_at_utc"], errors="coerce")
                if ts.notna().any():
                    chosen_idx = ts.idxmax()
                    chosen = df.loc[chosen_idx]
                    chosen_reason = f"latest created_at_utc={chosen.get('created_at_utc')}"

            if chosen is None:
                chosen = matches.iloc[0]
                chosen_reason = "first match (fallback)"

            print(
                f"WARNING: Multiple matches ({len(matches)}) for '{substring}'. "
                f"Using {chosen_reason}:\n"
                + "\n".join(f"  - {x}" for x in matches["experiment_name"].tolist())
            )
            return chosen

        globals()["find_exp"] = find_exp


_ensure_table4_context()

MODEL_COMPARISON = globals().get("MODEL_COMPARISON", [])
TABLE4_BASELINES = globals().get("TABLE4_BASELINES", [])
TABLE4_METRICS_3DP = globals().get("TABLE4_METRICS_3DP", [])
TABLE4_RAW_STATS_COLS = globals().get(
    "TABLE4_RAW_STATS_COLS",
    [
        "model_calls",
        "prompt_tokens_total",
        "output_tokens_total",
        "total_tokens_total",
        "total_tokens_avg",
        "latency_ms_p50",
    ],
)

print("TABLE 4: Model Comparison")
print("=" * 80)

rows = []
for model_name, exp_name in MODEL_COMPARISON:
    try:
        row = find_exp(df, exp_name)

        data = {"Model": model_name}

        # Main metrics
        for m in TABLE4_METRICS_3DP:
            val = row.get(m) if m in row.index else None
            data[m] = round(float(val), 3) if pd.notna(val) else None

        # Raw stats columns (copied verbatim from CSV; no processing)
        for c in TABLE4_RAW_STATS_COLS:
            data[c] = row.get(c) if c in row.index else None

        rows.append(data)

        ade = data.get("ade_model_avg")
        latency_ms_p50 = data.get("latency_ms_p50")
        try:
            latency_ms_p50_print = float(latency_ms_p50) if pd.notna(latency_ms_p50) else None
        except Exception:
            latency_ms_p50_print = None

        print(
            f"  {model_name}: ade={ade:.3f}"
            + (
                f", latency_p50={latency_ms_p50_print:.0f}ms"
                if latency_ms_p50_print is not None
                else ""
            )
            + f", matched={row.get('experiment_name')}"
        )

    except ValueError as e:
        print(f"  WARNING: {e}")

# Add baselines
for bname, exp_name in TABLE4_BASELINES:
    try:
        row = find_exp(df, exp_name)
        data = {"Model": bname}

        for m in TABLE4_METRICS_3DP:
            val = row.get(m) if m in row.index else None
            data[m] = round(float(val), 3) if pd.notna(val) else None

        # Raw stats columns (copied verbatim from CSV; no processing)
        for c in TABLE4_RAW_STATS_COLS:
            data[c] = row.get(c) if c in row.index else None

        rows.append(data)

    except ValueError as e:
        print(f"  WARNING for {bname}: {e}")

table_df = pd.DataFrame(rows)

# Reorder columns (keep only what exists)
col_order = ["Model"] + TABLE4_METRICS_3DP + TABLE4_RAW_STATS_COLS
col_order = [c for c in col_order if c in table_df.columns]

table_df = table_df[col_order]

table_df.to_csv(TABLE_OUT_DIR / "model_comparison.csv", index=False)
print(f"\nSaved: {TABLE_OUT_DIR / 'model_comparison.csv'}")
display(table_df)

## Summary

In [ ]:
# Print best results (robust to running this section standalone)
from pathlib import Path

import pandas as pd


def _load_task2_csv(csv_path: Path) -> pd.DataFrame:
    df_raw = pd.read_csv(csv_path)
    df = df_raw.dropna(subset=["experiment_name"])
    df = df[~df["experiment_name"].astype(str).str.startswith("--", na=False)]
    df = df[~df["experiment_name"].astype(str).str.startswith("-", na=False)]
    return df.reset_index(drop=True)


if "df" not in globals():
    candidates = [
        Path("../latex/source_data/task2_data.csv"),
        Path("latex/source_data/task2_data.csv"),
    ]
    csv_path = next((p for p in candidates if p.exists()), None)
    if csv_path is None:
        tried = ", ".join(str(p.resolve()) for p in candidates)
        raise FileNotFoundError(f"Missing CSV. Tried: {tried}")
    df = _load_task2_csv(csv_path)

if "find_exp" not in globals():

    def find_exp(df: pd.DataFrame, substring: str) -> pd.Series:
        matches = df[
            df["experiment_name"].astype(str).str.contains(substring, na=False, regex=False)
        ]
        if len(matches) == 0:
            raise ValueError(f"No match: '{substring}'")
        if len(matches) == 1:
            return matches.iloc[0]

        exact = matches[matches["experiment_name"] == substring]
        if len(exact) == 1:
            return exact.iloc[0]

        anchor_name = substring if substring.startswith("ANCHOR_") else f"ANCHOR_{substring}"
        anchor = matches[matches["experiment_name"] == anchor_name]
        if len(anchor) == 1:
            return anchor.iloc[0]

        if "created_at_utc" in matches.columns:
            ts = pd.to_datetime(matches["created_at_utc"], errors="coerce")
            if ts.notna().any():
                return df.loc[ts.idxmax()]

        return matches.iloc[0]

    globals()["find_exp"] = find_exp


print("=" * 80)
print("SUMMARY")
print("=" * 80)

vlm_df = df[df["selector"].isin(["gemini_genai", "openai_chat_completions"])]
if len(vlm_df) > 0:
    best = vlm_df.loc[vlm_df["ade_model_avg"].idxmin()]
    print("\nBest VLM result:")
    print(f"  Experiment: {best['experiment_name']}")
    print(f"  Selector: {best.get('selector', 'N/A')}")
    print(f"  ADE: {best['ade_model_avg']:.3f} m")
    if pd.notna(best.get("latency_ms_p50")):
        print(f"  Latency p50: {best['latency_ms_p50'] / 1000:.3f} s")

print("\nBaselines:")
for label, substr in [
    ("Planner Argmax", "policy=dummy_argmax"),
    ("Oracle (Min ADE)", "policy=oracle_min_ade"),
    ("Oracle (Min ADE, all)", "policy=oracle_min_ade_all"),
]:
    try:
        row = find_exp(df, substr)
        print(f"  {label}: ADE = {row['ade_model_avg']:.3f} m")
    except ValueError as e:
        print(f"  WARNING: {e}")